In [1]:
# author: Jen Epstein, using Chris Cane's ICE_Facilities notebook
# date: June 1 2026
# purpose: explore locations of ICE facilities wrt public water system service

In [2]:
import pandas as pd
import geopandas as gpd
#import urllib.request
#from zipfile import ZipFile
#import os

In [3]:
# import CWS boundaries from file
file = r"Service_Areas_V_3_0.gpkg"
cws = gpd.read_file(file, layer='CWS')
cws.head()

,Original_Data_Provider,Data_Provider_Type,Data_Source,PWSID,PWS_Name,Primacy_Agency,Pop_Cat_5,Population_Served_Count,Service_Connections_Count,Model_Method,...,Feature_Type,Method_Details,Verification_Status,Detailed_Facility_Report,Confirmed,PWSID_1,Shape_Length,Shape_Area,Date_Create,geometry
0,US EPA Office of Research and Development,Federal agency,,IA1103009,ALBERT CITY MUNI WATER SUPPLY,IA,"501-3,300",677.0,338.0,Random Forest,...,,,,https://echo.epa.gov/detailed-facility-report?...,,NaN,8439.583233,2.144797e+06,2024-06-06,"MULTIPOLYGON (((-94.95325 42.77116, -94.95325 ..."
1,US EPA Office of Research and Development,Federal agency,,IA1150030,MARATHON LIGHT & WATER DEPARTMENT,IA,<=500,230.0,120.0,Random Forest,...,,,,https://echo.epa.gov/detailed-facility-report?...,,NaN,7882.744094,2.905012e+06,2024-06-06,"MULTIPOLYGON (((-94.97816 42.85514, -94.97846 ..."
2,US EPA Office of Research and Development,Federal agency,,IA1175056,SIOUX RAPIDS WATER DEPARTMENT,IA,"501-3,300",920.0,373.0,Random Forest,...,,,,https://echo.epa.gov/detailed-facility-report?...,,NaN,12637.892945,3.471844e+06,2024-06-06,"MULTIPOLYGON (((-95.15328 42.88467, -95.15326 ..."
3,US EPA Office of Research and Development,Federal agency,,IA2109041,DICKENS WATER WORKS,IA,<=500,151.0,75.0,Random Forest,...,,,,https://echo.epa.gov/detailed-facility-report?...,,NaN,5576.325161,1.308007e+06,2024-06-06,"MULTIPOLYGON (((-95.02695 43.13076, -95.02701 ..."
4,US EPA Office of Research and Development,Federal agency,,IA2122067,FOSTORIA MUNICIPAL WATER SUPPLY,IA,<=500,233.0,133.0,Random Forest,...,,,,https://echo.epa.gov/detailed-facility-report?...,,NaN,6618.601702,1.983564e+06,2024-06-06,"MULTIPOLYGON (((-95.15316 43.23445, -95.15179 ..."


In [4]:
# import non-community system boundaries from file
file = r"Service_Areas_V_3_0.gpkg"
ncws = gpd.read_file(file, layer='T_NTNC')
ncws.head()

,PWSID,PWS_Name,LOCATION_CONFIDENCE,PRIMACY_AGENCY_CODE,PWS_TYPE_CODE,POPULATION_SERVED_COUNT,PRIMARY_SOURCE_CODE,IS_WHOLESALER_IND,IS_SCHOOL_OR_DAYCARE_IND,SERVICE_CONNECTIONS_COUNT,parcelnumb,AREAKM,SERVICE_AREA_TYPE,Detailed_Facility_Report,Data_Source,geometry
0,AK2212851,BIRCHWOOD RECREATION AND SHOOTING PARK,LOWER,AK,TNCWS,54.0,GW,N,N,2.0,NaN,0.125420,Restaurant,https://echo.epa.gov/detailed-facility-report?...,NaN,"MULTIPOLYGON (((-149.51042 61.41898, -149.5104..."
1,AK2310138,CHATANIKA LODGE,LOWER,AK,TNCWS,104.0,GW,N,N,1.0,NaN,0.045181,Restaurant,https://echo.epa.gov/detailed-facility-report?...,NaN,"MULTIPOLYGON (((-147.49856 65.1178, -147.50205..."
2,AK2111523,SHRINE OF ST. THERESE,LOWER,AK,TNCWS,78.0,SW,N,N,6.0,NaN,0.175184,Other Non-transient Area,https://echo.epa.gov/detailed-facility-report?...,NaN,"MULTIPOLYGON (((-134.78274 58.47236, -134.7827..."
3,AK2240634,4-LANDS BAR & LIQUOR STORE,LOWER,AK,TNCWS,100.0,GW,N,N,2.0,NaN,0.009208,Restaurant,https://echo.epa.gov/detailed-facility-report?...,NaN,"MULTIPOLYGON (((-151.37086 60.67203, -151.3708..."
4,AK2244256,CROOKED CREEK RV PARK,LOWER,AK,TNCWS,136.0,GW,N,N,148.0,NaN,0.002673,Recreation Area,https://echo.epa.gov/detailed-facility-report?...,NaN,"MULTIPOLYGON (((-151.28553 60.32043, -151.2855..."


In [5]:
# cws and ncws have different fields
print(cws.columns)
print(ncws.columns)

Index(['Original_Data_Provider', 'Data_Provider_Type', 'Data_Source', 'PWSID',
       'PWS_Name', 'Primacy_Agency', 'Pop_Cat_5', 'Population_Served_Count',
       'Service_Connections_Count', 'Model_Method', 'Service_Area_Type',
       'Symbology_Field', 'Modification_Method', 'Feature_Type',
       'Method_Details', 'Verification_Status', 'Detailed_Facility_Report',
       'Confirmed', 'PWSID_1', 'Shape_Length', 'Shape_Area', 'Date_Create',
       'geometry'],
      dtype='str')
Index(['PWSID', 'PWS_Name', 'LOCATION_CONFIDENCE', 'PRIMACY_AGENCY_CODE',
       'PWS_TYPE_CODE', 'POPULATION_SERVED_COUNT', 'PRIMARY_SOURCE_CODE',
       'IS_WHOLESALER_IND', 'IS_SCHOOL_OR_DAYCARE_IND',
       'SERVICE_CONNECTIONS_COUNT', 'parcelnumb', 'AREAKM',
       'SERVICE_AREA_TYPE', 'Detailed_Facility_Report', 'Data_Source',
       'geometry'],
      dtype='str')


In [6]:
# import detention facility locations obtained from DHS
file = r"ICE facilities - ICE.csv" # this is a copy of the Google sheets file downloaded 3 Jun 2026
detention_fac = pd.read_csv(file)
# convert to gdf
detention_fac = gpd.GeoDataFrame(detention_fac, geometry=gpd.points_from_xy(detention_fac.FAC_LONG, detention_fac.FAC_LAT), crs="EPSG:4326").to_crs(cws.crs)
detention_fac.head()

,fac,zip,retrieved from https://www.ice.gov/detention-facilities Feb 6 2026 though page says last updated June 2025,original_address,FAC_LAT,FAC_LONG,formatted,name,housenumber,street,...,country,country_code,confidence,confidence_city_level,confidence_street_level,confidence_building_level,attribution,attribution_license,attribution_url,geometry
0,Plymouth County Correctional FacilityBoston Fi...,2360.0,"26 Long Pond Road Plymouth, MA 02360","26 Long Pond Road Plymouth, MA 02360",41.932742,-70.657852,"26 Long Pond Road, Plymouth, MA 02360, United ...",NaN,26,Long Pond Road,...,United States,us,1.0,1.0,1.0,1.0,© OpenStreetMap contributors,Open Database License,https://www.openstreetmap.org/copyright,POINT (-70.65785 41.93274)
1,Wyatt Detention FacilityBoston Field Office950...,2863.0,"950 High Street Central Falls, RI 02863","950 High Street Central Falls, RI 02863",41.892811,-71.383200,"950 High Street, Central Falls, RI 02863, Unit...",NaN,950,High Street,...,United States,us,1.0,1.0,1.0,1.0,© OpenStreetMap contributors,Open Database License,https://www.openstreetmap.org/copyright,POINT (-71.3832 41.89281)
2,"Federal Correctional Institution - Berlin, NH ...",3570.0,"1 Success Loop Rd Berlin, NH 03570","1 Success Loop Rd Berlin, NH 03570",44.523499,-71.136130,"1 Success Loop Road, Berlin, NH 03570, United ...",NaN,1,Success Loop Road,...,United States,us,1.0,1.0,1.0,1.0,© OpenStreetMap contributors,Open Database License,https://www.openstreetmap.org/copyright,POINT (-71.13613 44.5235)
3,Strafford County CorrectionsBoston Field Offic...,3820.0,"266 County Farm Road Dover, NH 03820","266 County Farm Road Dover, NH 03820",43.219230,-70.938608,"266 County Farm Road, Dover, NH 03820, United ...",NaN,266,County Farm Road,...,United States,us,1.0,1.0,1.0,1.0,© OpenStreetMap contributors,Open Database License,https://www.openstreetmap.org/copyright,POINT (-70.93861 43.21923)
4,Cumberland County JailBoston Field Office50 Co...,4102.0,"50 County Way Portland, ME 04102","50 County Way Portland, ME 04102",43.652372,-70.281138,"50 County Way, Portland, ME 04102, United Stat...",NaN,50,County Way,...,United States,us,1.0,1.0,1.0,1.0,© OpenStreetMap contributors,Open Database License,https://www.openstreetmap.org/copyright,POINT (-70.28114 43.65237)


In [7]:
# join CWS to detention facilities that are within 
cws_detention = gpd.sjoin(detention_fac, cws, how='inner', predicate='within')
# look at counts for CWS
print("Number of detention facilities: ", detention_fac.shape[0])
print("Number of detention facilities within a CWS: ", cws_detention.shape[0])
print(f"Percentage of ICE detention facilities within a PWS: {(cws_detention.shape[0]/detention_fac.shape[0])*100:.0f}%")

Number of detention facilities:  142
Number of detention facilities within a CWS:  107
Percentage of ICE detention facilities within a PWS: 75%


In [8]:
# what types of water systems are the CWS? 
cws_detention.groupby("Service_Area_Type").count()['PWSID']

Service_Area_Type
Institution              2
Municipality            29
Other Residential        2
Residential Area        70
Secondary Residences     1
Wholesaler of Water      2
Name: PWSID, dtype: int64

In [9]:
cws_detention.groupby("Service_Area_Type").count()['PWSID'].sum()

np.int64(106)

In [10]:
# missing info for 1 facility

In [11]:
# join NCWS to detention facilities that are within 
ncws_detention = gpd.sjoin(detention_fac, ncws, how='inner', predicate='within')
# look at counts for CWS
print("Number of detention facilities: ", detention_fac.shape[0])
print("Number of detention facilities within a NCWS: ", ncws_detention.shape[0])
print(f"Percentage of ICE detention facilities within a PWS: {(ncws_detention.shape[0]/detention_fac.shape[0])*100:.0f}%")

Number of detention facilities:  142
Number of detention facilities within a NCWS:  2
Percentage of ICE detention facilities within a PWS: 1%


In [12]:
# what types of water systems are thes NCWS?
ncws_detention.groupby("SERVICE_AREA_TYPE").count()['PWSID']

SERVICE_AREA_TYPE
Other Residential       1
Other Transient Area    1
Name: PWSID, dtype: int64

In [13]:
# ECHO facility report links
#for x in cws_detention['Detailed_Facility_Report']: print(x)
#for x in ncws_detention['Detailed_Facility_Report']: print(x)

In [14]:
# import and join sdwa enforcement data -- if needed -- next update 7/15/2026
#url = r"https://echo.epa.gov/files/echodownloads/SDWA_latest_downloads.zip"
#file_name = "SDWA_latest_downloads.zip"
#urllib.request.urlretrieve(url, file_name)
# Extract the downloaded ZIP file
#with ZipFile(file_name, 'r') as zip_file:
#   zip_file.extractall()
#   print("Files extracted successfully")

In [15]:
# import sdwa system data from file
file = r"SDWA_PUB_WATER_SYSTEMS.csv"
sdwa_systems = gpd.read_file(file, encoding="utf8")

In [16]:
# merge with pws boundary gdfs
cws_detention = cws_detention.merge(sdwa_systems, on='PWSID')
ncws_detention = ncws_detention.merge(sdwa_systems, on='PWSID')

In [17]:
# look at counts to see if join checks out
print("Number of CWS containing detention facilities with SDWA facility data: ", cws_detention.shape[0])
print("Number of NCWS containing detention facilities with SDWA facility data: ", ncws_detention.shape[0])

Number of CWS containing detention facilities with SDWA facility data:  107
Number of NCWS containing detention facilities with SDWA facility data:  2


In [ ]:
# joins worked for all the water systems containing detention facilities 
# now we're ready to look at SDWA facility info

In [ ]:
sdwa_systems.columns